# Tutorial 04: Middleware and Filters - Production-Ready Safeguards

##  Learning Objectives
After completing this notebook, you will be able to:
- Understand the difference between middleware and filters in the Agent Framework
- Implement content filtering for safety and compliance
- Add logging and observability to agents
- Transform requests and responses in real-time
- Build production-ready safety mechanisms
- Handle rate limiting and error recovery

##  Key Concepts

### Middleware vs Filters: What's the Difference?

**Middleware:**
- Wraps the entire agent execution
- Can modify requests before they reach the agent
- Can transform responses after the agent responds
- Examples: Authentication, rate limiting, logging

**Filters:**
- Focus on content safety and compliance
- Block harmful or inappropriate content
- Examples: Profanity filter, PII detection, content moderation

### Production Concerns

Real-world agents need:
1. **Safety**: Content filtering, blocking harmful requests
2. **Observability**: Logging, metrics, tracing
3. **Reliability**: Error handling, retry logic, circuit breakers
4. **Compliance**: Data privacy, audit trails
5. **Performance**: Caching, rate limiting

---

## Step 1: Setup and Imports

In [ ]:
import asyncio
import json
import re
import time
from collections.abc import MutableSequence, Sequence, Callable, Awaitable
from datetime import datetime
from typing import Annotated, Any
from random import choice, randint

from agent_framework import (
    Agent,
    Message,
    ChatMiddleware,
    ChatOptions,
    ChatResponse,
    ChatContext,
)
from agent_framework.azure import AzureAIAgentClient
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity.aio import AzureCliCredential
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()
print("✅ Import successful!")

In [ ]:
import os
from agent_framework.azure import AzureOpenAIChatClient

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication Method Selection ===
# True: API Key Auth, False: DefaultAzureCredential Auth
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 Auth method: API Key")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework auto-loads the AZURE_OPENAI_API_KEY environment variable
    # and prioritizes it over credential, so we explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 Auth method: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: AzureAIAgentClient is an Async Context Manager.
# When using Agent with `async with`, the client is also entered/closed internally.
# Therefore, reusing a globally created AzureAIAgentClient across multiple cells/demos
# can easily cause "HTTP transport has already been closed" errors.
# In the following demos, we use the approach of "creating a new client each time".
def create_foundry_chat_client(credential):
    return AzureAIAgentClient(
        credential=credential,
        project_endpoint=project_endpoint,
        model_deployment_name=model_deployment,
    )

def create_azure_openai_chat_client():
    # Create Azure OpenAI client
    return AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )


print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Step 2: The Problem - An Unsafe Production Agent

Let's first see what happens without any safety mechanisms.

In [ ]:
async def unsafe_agent_demo():
    """
    Demonstrates an agent without safety filters.
    In production, this is dangerous!
    """
    print("=== Unsafe Agent (No Filters) ===")
    print("This agent has no safety mechanisms - not suitable for production!\n")
    
    async with Agent(
        client=create_azure_openai_chat_client(),
        name="UnsafeTravelAgent",
        instructions="You are a travel assistant. Answer all travel-related questions.",
    ) as agent:
        session = agent.create_session()
        
        # Simulate potentially problematic requests
        test_queries = [
            "What is the weather in Paris?",  # Normal query
            "My email is john.doe@company.com. Can you help me plan a trip?",  # PII exposure
            "Tell me about the destination, you stupid bot!",  # Profanity
        ]
        
        for query in test_queries:
            print(f"User: {query}")
            response = await agent.run(query, session=session)
            print(f"Agent: {response.text}\n")
        
        print(" Problems with this approach:")
        print("- No content filtering for profanity")
        print("- No PII (email) detection or masking")
        print("- No logging for audit/compliance")
        print("- No rate limiting or abuse prevention")
        print("- No error recovery mechanisms\n")

await unsafe_agent_demo()

## Step 3: Content Safety Filter

Let's build our first middleware - a content safety filter.

In [ ]:
class ContentSafetyMiddleware(ChatMiddleware):
    """
    Middleware that filters inappropriate content in both requests and responses.
    
    This middleware:
    1. Checks user messages for profanity/inappropriate content
    2. Blocks requests that violate content policies
    3. Can also filter agent responses if needed
    """
    
    def __init__(self):
        # Simple profanity list (use proper services in production)
        self.blocked_words = {
            "stupid", "idiot", "fool", "useless", 
            "hate", "kill", "die", "bomb"
        }
        
        # Inappropriate request patterns
        self.blocked_patterns = [
            r"hack\s+into",
            r"illegal\s+activities?",
            r"how\s+to\s+hurt",
        ]
    
    def _contains_inappropriate_content(self, text: str) -> tuple[bool, str | None]:
        """Checks if text contains inappropriate content."""
        text_lower = text.lower()
        
        # Check for blocked words
        for word in self.blocked_words:
            if word in text_lower:
                return True, f"Inappropriate language detected: '{word}'"
        
        # Check for blocked patterns
        for pattern in self.blocked_patterns:
            if re.search(pattern, text_lower):
                return True, f"Inappropriate request pattern detected"
        
        return False, None
    
    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Process messages through the content safety filter."""
        
        # Filter incoming user messages
        for message in context.messages:
            if message.role == "user":
                # Check message text
                text = message.text
                if text:
                    is_inappropriate, reason = self._contains_inappropriate_content(text)
                    if is_inappropriate:
                        print(f"🚫 Content blocked: {reason}")
                        # Set result to override execution
                        context.result = ChatResponse(
                            messages=[
                                Message(
                                    role="assistant",
                                    contents=["I'm sorry, but I cannot process requests containing inappropriate language or content. Please rephrase your message politely."],
                                )
                            ]
                        )
                        return  # Don't call call_next()
        
        print("✅ Content safety check passed")
        
        # Continue to next middleware or chat client
        await call_next()

print(" Content Safety Middleware created!")

## Step 4: PII Detection and Masking Middleware

Let's add privacy protection by detecting and masking personally identifiable information.

In [ ]:
class PIIDetectionMiddleware(ChatMiddleware):
    """
    Middleware that detects and masks Personally Identifiable Information (PII).

    This middleware:
    1. Detects emails, phone numbers, credit cards, etc.
    2. Masks sensitive data before sending to AI
    3. Logs PII detection events for compliance
    """

    def __init__(self):
        # PII detection patterns
        self.pii_patterns = {
            "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
            "phone": r"\b(?:\+?1[-.]?)?\(?([0-9]{3})\)?[-.]?([0-9]{3})[-.]?([0-9]{4})\b",
            "ssn": r"\b\d{3}-\d{2}-\d{4}\b",
            "credit_card": r"\b(?:\d{4}[-\s]?){3}\d{4}\b",
        }

    def _mask_pii(self, text: str) -> tuple[str, list[str]]:
        """Masks PII in text and returns masked text + detected PII types."""
        masked_text = text
        detected_pii = []

        for pii_type, pattern in self.pii_patterns.items():
            matches = re.finditer(pattern, text)
            for match in matches:
                detected_pii.append(pii_type)
                # Mask with type-specific replacement
                if pii_type == "email":
                    replacement = "[EMAIL_MASKED]"
                elif pii_type == "phone":
                    replacement = "[PHONE_MASKED]"
                elif pii_type == "ssn":
                    replacement = "[SSN_MASKED]"
                elif pii_type == "credit_card":
                    replacement = "[CARD_MASKED]"
                else:
                    replacement = "[PII_MASKED]"

                masked_text = masked_text.replace(match.group(), replacement)

        return masked_text, list(set(detected_pii))  # Remove duplicates

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Process messages through PII detection and masking."""

        # Process PII in user messages
        for message in context.messages:
            if message.role == "user":
                for content in message.contents:
                    if hasattr(content, "text") and content.text:
                        original_text = content.text
                        masked_text, detected_pii = self._mask_pii(original_text)

                        if detected_pii:
                            print(
                                f"🔒 PII detected and masked: {', '.join(detected_pii)}"
                            )
                            # Update message content with masked version
                            content.text = masked_text

                            # In production, log this event for compliance
                            print(
                                f"📋 Compliance log: PII types {detected_pii} detected at {datetime.now()}"
                            )

        print("✅ PII detection check complete")

        # Continue to next middleware or chat client
        await call_next()


print(" PII Detection Middleware created!")

## Step 5: Logging and Observability Middleware

Essential for production: comprehensive logging and monitoring.

In [ ]:
class LoggingMiddleware(ChatMiddleware):
    """
    Middleware that provides comprehensive logging and observability.
    
    This middleware:
    1. Logs all requests and responses
    2. Tracks processing time and performance metrics
    3. Records errors and exceptions
    4. Provides audit trails for compliance
    """
    
    def __init__(self):
        self.request_count = 0
        self.total_processing_time = 0
    
    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Process messages with comprehensive logging."""
        
        self.request_count += 1
        start_time = time.time()
        timestamp = datetime.now().isoformat()
        
        messages = context.messages or []
        
        # Log incoming request
        user_messages = [msg for msg in messages if msg.role == "user"]
        
        print(f"📥 [{timestamp}] Request #{self.request_count}")
        print(f"   Messages: {len(messages)} total, {len(user_messages)} from user")
        
        # Extract user message preview (first 100 characters)
        for msg in user_messages:
            text = msg.text
            if text:
                preview = text[:100] + "..." if len(text) > 100 else text
                print(f"   User input: {preview}")
        
        try:
            # Continue to next middleware or chat client
            await call_next()
            
            # Calculate processing time
            processing_time = time.time() - start_time
            self.total_processing_time += processing_time
            
            # Log successful response
            response_preview = ""
            if context.result and hasattr(context.result, "messages") and context.result.messages:
                first_msg = context.result.messages[0]
                text = first_msg.text
                if text:
                    response_preview = text[:100] + "..." if len(text) > 100 else text
            
            print(f"📤 [{timestamp}] Response #{self.request_count}")
            print(f"   Processing time: {processing_time:.2f}s")
            print(f"   Status: Success")
            print(f"   Response preview: {response_preview}")
            print(f"   Average processing time: {self.total_processing_time / self.request_count:.2f}s")
            
        except Exception as e:
            # Log error
            processing_time = time.time() - start_time
            print(f"❌ [{timestamp}] Request #{self.request_count}  Error")
            print(f"   Processing time: {processing_time:.2f}s")
            print(f"   Error type: {type(e).__name__}")
            print(f"   Error message: {str(e)}")
            
            # Re-raise the exception
            raise

print(" Logging Middleware created!")

## Step 6: Rate Limiting Middleware

Prevent abuse and manage resource usage.

In [ ]:
class RateLimitingMiddleware(ChatMiddleware):
    """
    Middleware that implements rate limiting to prevent abuse.
    
    This middleware:
    1. Tracks requests per time window
    2. Blocks requests that exceed limits
    3. Provides different limits for different user types
    4. Implements sliding window rate limiting
    """
    
    def __init__(self, requests_per_minute: int = 10, window_size: int = 60):
        self.requests_per_minute = requests_per_minute
        self.window_size = window_size  # seconds
        self.request_times = []  # List of request timestamps
    
    def _clean_old_requests(self):
        """Remove request timestamps outside the current window."""
        current_time = time.time()
        cutoff_time = current_time - self.window_size
        self.request_times = [
            req_time for req_time in self.request_times 
            if req_time > cutoff_time
        ]
    
    def _is_rate_limited(self) -> tuple[bool, int]:
        """Check if the current request should be rate limited."""
        self._clean_old_requests()
        
        current_requests = len(self.request_times)
        is_limited = current_requests >= self.requests_per_minute
        
        return is_limited, current_requests
    
    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Process messages through rate limiting."""
        
        # Check rate limit
        is_limited, current_requests = self._is_rate_limited()
        
        if is_limited:
            print(f"🚦 Rate limit exceeded: {current_requests}/{self.requests_per_minute} requests/min")
            context.result = ChatResponse(
                messages=[
                    Message(
                        role="assistant",
                        text=f"Sorry, you have reached the rate limit of {self.requests_per_minute} requests per minute."
                        f"Please wait a moment before sending your next message."
                    )
                ]
            )
            return  # Don't call call_next()
        
        # Record this request
        self.request_times.append(time.time())
        
        remaining_requests = self.requests_per_minute - current_requests - 1
        print(f"✅ Rate limit check passed ( {remaining_requests} requests remaining)")
        
        # Continue to next middleware or chat client
        await call_next()

print(" Rate Limiting Middleware created!")

## Step 7: Production-Ready Agent with All Middleware

Let's combine all middleware into a production-ready agent!

In [ ]:
# Simple travel tools for demo
def get_weather(location: Annotated[str, Field(description="City or country name")]) -> str:
    """Returns the current weather for the specified location."""
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy"]
    temp = randint(15, 32)
    return f"Weather in {location}: {choice(conditions)}, {temp}°C"


def book_hotel(
    location: Annotated[str, Field(description="City name or location")],
    checkin: Annotated[str, Field(description="Check-in date (YYYY-MM-DD)")],
    nights: Annotated[int, Field(description="Number of nights")],
) -> str:
    """Creates a hotel reservation."""
    return (
        f"Hotel booked in {location} from {checkin} for {nights} nights. "
        f"Confirmation number: HTL{randint(1000,9999)}"
    )


print(" Travel tools defined")

In [ ]:
async def production_safe_agent():
    """
    Demonstrates a production-ready agent with comprehensive middleware.
    """
    print("=== Production-Ready Agent with Middleware ===")
    print(
        "This agent includes: content filtering, PII masking, logging, rate limiting\n"
    )

    # Create all middleware instances
    content_filter = ContentSafetyMiddleware()
    pii_detector = PIIDetectionMiddleware()
    logger = LoggingMiddleware()
    rate_limiter = RateLimitingMiddleware(requests_per_minute=5)  # Low limit for demo

    chat_client = create_azure_openai_chat_client()

    # Create agent with middleware stack
    # Middleware executes in the order provided
    async with Agent(
        client=chat_client,
        name="ProductionTravelAgent",
        instructions="""
        You are a professional travel assistant.
        Provide helpful, polite, and clear responses.
        Always prioritize user safety and privacy.
        """,
        tools=[get_weather, book_hotel],
        middleware=[
            rate_limiter,  # Check rate limit first
            logger,  # Log everything
            content_filter,  # Filter inappropriate content
            pii_detector,  # Mask PII
        ],
    ) as agent:
        session = agent.create_session()

        # Test scenarios
        test_scenarios = [
            # 1. Normal request
            "What is the weather like in Tokyo?",

            # 2. Request with PII (will be masked)
            "I am planning a trip to Paris. Contact me at sarah.jones@example.com if needed.",

            # 3. Inappropriate content (will be blocked)
            "You are useless! Tell me about Rome.",

            # 4. Normal booking request
            "Book a hotel in Barcelona from 2024-12-01 for 3 nights.",

            # 5. Another normal request (may hit rate limit)
            "What are some recommended restaurants in Barcelona?",

            # 6. One more to test rate limiting
            "Tell me about tourist attractions in Barcelona.",
        ]

        for i, query in enumerate(test_scenarios, 1):
            print(f"\n{'='*60}")
            print(f"Test Scenario {i}")
            print(f"{'='*60}")
            print(f"User: {query}")
            print("\n--- Processing through Middleware ---")

            try:
                response = await agent.run(query, session=session)
                print("\n--- Final Response ---")
                print(f"Agent: {response.text}")

            except Exception as e:
                print(f"\n❌ Error: {e}")

            # Short delay between requests
            await asyncio.sleep(1)

        print(f"\n{'='*60}")
        print("✅ Production safety demo complete!")
        print(f"{'='*60}")
        print("Middleware successfully:")
        print("✅ Filtered inappropriate content")
        print("✅ Detected and masked PII (email addresses)")
        print("✅ Logged all requests and responses")
        print("✅ Enforced rate limiting")
        print("✅ Provided comprehensive audit trails")


await production_safe_agent()

## Step 8: Response Transformation Middleware

Sometimes you need to transform or enhance agent responses.

In [ ]:
class ResponseTransformMiddleware(ChatMiddleware):
    """
    Middleware that transforms agent responses.

    This middleware:
    1. Adds disclaimers to responses
    2. Formats responses for readability
    3. Adds metadata and branding
    4. Implements response caching
    """

    def __init__(self, add_disclaimers: bool = True, add_branding: bool = True):
        self.add_disclaimers = add_disclaimers
        self.add_branding = add_branding
        self.response_cache = {}  # Simple in-memory cache

    def _add_travel_disclaimer(self, response_text: str) -> str:
        """Adds travel-specific disclaimers to responses."""
        disclaimer = (
            "\n\n📋 *Note: Travel conditions and requirements are subject to change."
            "Always check official sources for the latest information before traveling.*"
        )
        return response_text + disclaimer

    def _add_branding(self, response_text: str) -> str:
        """Adds company branding to responses."""
        branding = "\n\n *Powered by SafeTravel AI Assistant*"
        return response_text + branding

    def _format_response(self, response_text: str) -> str:
        """Improves response formatting."""
        # Guard against None (text can be None depending on SDK/model/tool responses)
        if response_text is None:
            return ""

        # Add emoji icons for better visual appeal
        replacements = {
            "weather": "☀️ weather",
            "temperature": "🌡 temperature",
            "hotel": "🏨 hotel",
            "booking": "📅 booking",
            "confirmation": "🔖 confirmation",
        }

        formatted_text = response_text
        for old, new in replacements.items():
            formatted_text = formatted_text.replace(old, new)

        return formatted_text

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Process messages and transform responses."""

        # Continue to next middleware or chat client
        await call_next()

        # If there is a result, transform each response message
        if context.result and hasattr(context.result, "messages"):
            for message in context.result.messages:
                if hasattr(message, "contents") and message.contents:
                    for content in message.contents:
                        if hasattr(content, "text") and content.text is not None:
                            original_text = content.text
                            transformed_text = original_text

                            # Apply transformations
                            transformed_text = self._format_response(transformed_text)

                            if self.add_disclaimers:
                                transformed_text = self._add_travel_disclaimer(
                                    transformed_text
                                )

                            if self.add_branding:
                                transformed_text = self._add_branding(transformed_text)

                            # Update content
                            content.text = transformed_text

                            print(
                                "✅ Response transformation complete (formatting, disclaimers, branding added)"
                            )


print(" Response Transformation Middleware created!")

## Step 9: Complete Production Pipeline

Let's put it all together - a complete production pipeline with all safeguards.

In [ ]:
async def complete_production_pipeline():
    """
    Demonstrates a complete production middleware pipeline.
    """
    print("=== Complete Production Pipeline ===")
    print(
        "Full middleware stack: Rate Limiting → Logging → Content Safety → PII Detection → Response Transformation\n"
    )

    # Create complete middleware stack
    rate_limiter = RateLimitingMiddleware(requests_per_minute=3)  # Very low for demo
    logger = LoggingMiddleware()
    content_filter = ContentSafetyMiddleware()
    pii_detector = PIIDetectionMiddleware()
    response_transformer = ResponseTransformMiddleware()

    chat_client = create_azure_openai_chat_client()

    async with Agent(
        client=chat_client,
        name="EnterpriseTravelAgent",
        instructions="""
        You are an enterprise-quality travel assistant.
        Provide accurate, helpful, and safe travel information.
        Always maintain a professional and polite demeanor.
        """,
        tools=[get_weather, book_hotel],
        middleware=[
            rate_limiter,  # 1. Check rate limit first
            logger,  # 2. Log all activity
            content_filter,  # 3. Filter inappropriate content
            pii_detector,  # 4. Detect and mask PII
            response_transformer,  # 5. Transform responses (runs after agent)
        ],
    ) as agent:
        session = agent.create_session()

        # Comprehensive test scenarios
        scenarios = [
            {
                "name": "Normal Weather Query",
                "query": "What is the weather in London today?",
                "expected": "Should work normally with enhanced formatting",
            },
            {
                "name": "PII Detection Test",
                "query": "I am traveling to Paris. Contact me at john.smith@email.com or 555-123-4567.",
                "expected": "PII should be detected and masked",
            },
            {
                "name": "Content Filter Test",
                "query": "You are useless! Tell me about Rome.",
                "expected": "Should be blocked by content filter",
            },
            {
                "name": "Rate Limit Test",
                "query": "How about Barcelona?",
                "expected": "May hit rate limit (max 3 requests)",
            },
        ]

        for i, scenario in enumerate(scenarios, 1):
            print(f"\n{'='*80}")
            print(f"Scenario {i}: {scenario['name']}")
            print(f"Expected result: {scenario['expected']}")
            print(f"{'='*80}")
            print(f"User: {scenario['query']}")
            print("\n--- Processing through middleware stack ---")

            try:
                response = await agent.run(scenario["query"], session=session)
                print("\n--- Final Enhanced Response ---")
                print(f"Agent: {response.text}")

            except Exception as e:
                print(f"\n❌ Pipeline error: {e}")

            # Delay between requests
            if i < len(scenarios):
                print("\n⏳ Waiting 2 seconds before next request...")
                await asyncio.sleep(2)

        print(f"\n{'='*80}")
        print("🏆 Enterprise production pipeline complete!")
        print(f"{'='*80}")
        print("\n🛡 Security features demonstrated:")
        print("   ✅ Rate limiting (3 requests/min)")
        print("   ✅ Comprehensive request/response logging")
        print("   ✅ Content safety filtering")
        print("   ✅ PII detection and masking")
        print("   ✅ Response transformation and branding")
        print("   ✅ Error handling and recovery")
        print("   ✅ Audit trails for compliance")
        print("\n✅ This agent is production-ready!")


await complete_production_pipeline()

##  Understanding Middleware Architecture

### Middleware Execution Order

Middleware executes in a "pipeline" pattern:

```python
# Requests pass through middleware in order:
User Input
    ↓
Rate Limiter     # Check limits first
    ↓
Logger           # Log everything
    ↓
Content Filter   # Safety check
    ↓
PII Detector     # Privacy protection
    ↓
Agent Execution  # Core AI processing
    ↓
Response Transform # Output formatting
    ↓
Logger           # Log response
    ↓
Final Response
```

### Key Patterns

**1. Early Termination**
- Middleware can stop the pipeline early
- Example: Rate limiter blocks excessive requests
- Example: Content filter rejects inappropriate content

**2. Request Transformation**
- Modify messages before they reach the agent
- Example: PII masking replaces sensitive data

**3. Response Enhancement**
- Transform agent responses before returning
- Example: Add disclaimers and formatting

**4. Cross-Cutting Concerns**
- Logging, monitoring, observability
- Applied consistently to all requests

### Production Considerations

| Middleware Type | Purpose | Production Features |
|----------------|---------|---------------------|  
| **Rate Limiting** | Abuse prevention | Redis-based counters, per-user limits |
| **Content Safety** | Block harmful content | AI-powered detection, custom rules |
| **PII Detection** | Privacy compliance | ML-based detection, data residency |
| **Logging** | Observability | Structured logs, distributed tracing |
| **Caching** | Performance | Redis/Memcached, TTL policies |
| **Authentication** | Security | OAuth, API keys, role-based access |

##  Key Takeaways

### What You Learned

1. **Middleware Architecture**
   - Pipeline pattern for request/response processing
   - Early termination for safety and limits
   - Cross-cutting concerns (logging, monitoring)

2. **Production Safety**
   - Content filtering prevents inappropriate interactions
   - PII detection protects user privacy
   - Rate limiting prevents abuse
   - Comprehensive logging enables debugging

3. **Enterprise Features**
   - Response transformation for consistency
   - Audit trails for compliance
   - Error handling and recovery
   - Performance monitoring

### Best Practices

1. **Layer Security** - Multiple middleware layers provide defense in depth
2. **Fail Fast** - Check limits and safety early in the pipeline
3. **Log Everything** - Comprehensive logging for debugging and compliance
4. **Transform Carefully** - Enhance output while preserving original context
5. **Monitor Performance** - Track processing time and bottlenecks

### Production Patterns

```python
# Enterprise middleware stack
middleware_stack = [
    AuthenticationMiddleware(),     # Verify identity
    RateLimitingMiddleware(),      # Prevent abuse
    LoggingMiddleware(),           # Audit everything
    ContentSafetyMiddleware(),     # Safety first
    PIIDetectionMiddleware(),      # Privacy protection
    CachingMiddleware(),           # Optimize performance
    ResponseTransformMiddleware(), # Consistent formatting
]

# Agent with production middleware
agent = Agent(
    client=client,
    middleware=middleware_stack
)
```

##  Exercises

1. **Custom Content Filter** - Create a middleware that blocks requests about sensitive topics (politics, medical advice)
2. **Performance Monitor** - Build a middleware that tracks and alerts on slow responses
3. **A/B Testing** - Create a middleware that randomly selects different agent configurations
4. **Caching Layer** - Implement response caching to improve performance for repeated queries
5. **Multilingual Support** - Build a middleware that detects language and routes to appropriate specialized agents

In [ ]:
# Exercise: Create a performance monitoring middleware

class PerformanceMonitorMiddleware(ChatMiddleware):
    """
    Monitors agent performance and alerts on slow responses.
    
    Tasks:
    1. Track response times
    2. Calculate moving averages
    3. Alert when responses are unusually slow
    4. Provide performance metrics
    """
    
    def __init__(self, slow_threshold_seconds: float = 5.0):
        self.slow_threshold = slow_threshold_seconds
        # Your code here!
        pass
    
    async def process(self, context, call_next):
        # Implement here!
        # Remember to:
        # - Measure processing time
        # - Track statistics
        # - Alert on slow responses
        # - Call await call_next() to continue the pipeline
        pass

# Test your middleware here!
print(" Exercise ready - Implement the PerformanceMonitorMiddleware!")

##  Next Steps

Congratulations! You now have a production-ready agent with comprehensive safeguards.

But enterprise applications need more:
-  No multi-agent coordination
-  No workflow orchestration
-  No human-in-the-loop approvals

---

### Quick Reference

**Create Middleware:**
```python
class MyMiddleware(ChatMiddleware):
    async def process(self, context, call_next):
        # Process request
        await call_next()
        # Transform response
```

**Use with Agent:**
```python
agent = Agent(
    client=client,
    middleware=[MyMiddleware()]
)
```

**Early Termination:**
```python
# Block request without calling agent
context.result = ChatResponse(
    messages=[Message(role="assistant", contents=["Request blocked"])]
)
return  # Don't call call_next()
```